# MVC Project: Multilayer Perceptron
### Task 7 — Python Implementation & GitHub
**Artificial Neural Networks**

---


## Load MNIST Dataset

MNIST can be loaded in one line using the code snippet below. It downloads automatically on the first run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import urllib.request, os

# Download MNIST if not already present
url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
if not os.path.exists("mnist.npz"):
    print("Downloading MNIST...")
    urllib.request.urlretrieve(url, "mnist.npz")
    print("Download complete.")

data = np.load('mnist.npz')
X_train = data['x_train'].reshape(-1, 784) / 255.0  # (60000, 784)
Y_train = data['y_train']                             # (60000,)
X_test  = data['x_test'].reshape(-1, 784) / 255.0   # (10000, 784)
Y_test  = data['y_test']                              # (10000,)

# Convert labels to one-hot vectors for the loss calculation
def to_one_hot(labels, num_classes=10):
    m = labels.shape[0]
    one_hot = np.zeros((m, num_classes))
    one_hot[np.arange(m), labels] = 1
    return one_hot

Y_train_oh = to_one_hot(Y_train)  # (60000, 10)
Y_test_oh  = to_one_hot(Y_test)   # (10000, 10)

print(f"X_train: {X_train.shape}   Y_train: {Y_train.shape}")
print(f"X_test:  {X_test.shape}   Y_test:  {Y_test.shape}")


## Network Architecture

| Layer | Type | Neurons | Activation |
|-------|------|---------|------------|
| Layer 0 | Input | 784 | — |
| Layer 1 | Hidden layer 1 | 128 | Sigmoid |
| Layer 2 | Hidden layer 2 | 64 | Sigmoid |
| Layer 3 | Output | 10 | Sigmoid |

Initialise all weights randomly in the range [−0.5, 0.5] and all biases to zero. Use `np.random.seed(42)` for reproducibility.


In [ ]:
np.random.seed(42)

# Weight matrices
W1 = np.random.uniform(-0.5, 0.5, (784, 128))   # Input → Hidden 1
b1 = np.zeros((1, 128))

W2 = np.random.uniform(-0.5, 0.5, (128, 64))    # Hidden 1 → Hidden 2
b2 = np.zeros((1, 64))

W3 = np.random.uniform(-0.5, 0.5, (64, 10))     # Hidden 2 → Output
b3 = np.zeros((1, 10))

print("Weight shapes:")
print(f"  W1: {W1.shape},  b1: {b1.shape}")
print(f"  W2: {W2.shape},  b2: {b2.shape}")
print(f"  W3: {W3.shape},  b3: {b3.shape}")


---
## Step 1 — Sigmoid Activation Function *(from Task 1)*

**Implement sigmoid and its derivative**

You derived the sigmoid derivative yourself in Task 3. Now implement both:


In [ ]:
def sigmoid(z):
    # Hint: use np.clip(z, -500, 500) to avoid overflow
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_derivative(a):
    # Hint: you derived this in Task 3 - express it in terms of a
    # sigma'(z) = sigma(z) * (1 - sigma(z)) = a * (1 - a)
    return a * (1.0 - a)


---
## Step 2 — Forward Pass *(from Task 1)*

**Implement the forward pass**

Compute the weighted sum and activation for each layer. Store all intermediate values — you will need them in backpropagation.


In [ ]:
def forward_pass(X, W1, b1, W2, b2, W3, b3):
    # Layer 1
    Z1 = X @ W1 + b1      # weighted sum: X @ W1 + b1
    A1 = sigmoid(Z1)       # sigmoid activation

    # Layer 2
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    # Output layer
    Z3 = A2 @ W3 + b3
    A3 = sigmoid(Z3)

    return Z1, A1, Z2, A2, Z3, A3


---
## Step 3 — MSE Loss *(from Task 2)*

**Implement the MSE loss function**


In [ ]:
def mse_loss(Y_true, Y_pred):
    # Y_true: one-hot matrix (m, 10)
    # Y_pred: network output (m, 10)
    # Return: a single scalar value
    return np.mean(np.sum((Y_true - Y_pred) ** 2, axis=1))


---
## Step 4 — Backpropagation *(from Task 3)*

**Implement backpropagation using the delta method**

Follow the exact same steps you computed by hand in Task 3B. Use the delta notation you are already familiar with.


In [ ]:
def backpropagation(X, Y_true, Z1, A1, Z2, A2, Z3, A3,
                    W1, W2, W3):
    m = X.shape[0]

    # Output layer delta
    delta3 = -2 * (Y_true - A3) * sigmoid_derivative(A3)

    # Gradients for W3, b3
    dW3 = (A2.T @ delta3) / m
    db3 = np.sum(delta3, axis=0, keepdims=True) / m

    # Hidden layer 2 delta
    delta2 = (delta3 @ W3.T) * sigmoid_derivative(A2)

    # Gradients for W2, b2
    dW2 = (A1.T @ delta2) / m
    db2 = np.sum(delta2, axis=0, keepdims=True) / m

    # Hidden layer 1 delta
    delta1 = (delta2 @ W2.T) * sigmoid_derivative(A1)

    # Gradients for W1, b1
    dW1 = (X.T @ delta1) / m
    db1 = np.sum(delta1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2, dW3, db3


---
## Step 5 — Weight Update *(from Task 4)*

**Implement the gradient descent update rule**


In [ ]:
def update_weights(W1, b1, W2, b2, W3, b3,
                   dW1, db1, dW2, db2, dW3, db3,
                   learning_rate):
    # Apply: W = W - learning_rate * dW
    #        b = b - learning_rate * db
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

    W3 = W3 - learning_rate * dW3
    b3 = b3 - learning_rate * db3

    # Return all updated weights and biases
    return W1, b1, W2, b2, W3, b3


---
## Training Loop

Once all five functions are implemented, combine them into a training loop. Use mini-batch gradient descent with batch size 32 as you studied in Task 5.

**Complete the training loop**


In [ ]:
# Re-initialise weights before training
np.random.seed(42)
W1 = np.random.uniform(-0.5, 0.5, (784, 128));  b1 = np.zeros((1, 128))
W2 = np.random.uniform(-0.5, 0.5, (128, 64));   b2 = np.zeros((1, 64))
W3 = np.random.uniform(-0.5, 0.5, (64, 10));    b3 = np.zeros((1, 10))

learning_rate = 0.1
epochs = 20
batch_size = 32
loss_history = []

for epoch in range(epochs):
    # Shuffle training data
    idx = np.random.permutation(X_train.shape[0])
    X_shuf = X_train[idx]
    Y_shuf = Y_train_oh[idx]  # one-hot labels

    # Mini-batch loop
    for start in range(0, X_train.shape[0], batch_size):
        X_batch = X_shuf[start : start + batch_size]
        Y_batch = Y_shuf[start : start + batch_size]

        # Step 1: Forward pass
        Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_batch, W1, b1, W2, b2, W3, b3)

        # Step 2: Compute loss
        loss = mse_loss(Y_batch, A3)

        # Step 3: Backpropagation
        dW1, db1, dW2, db2, dW3, db3 = backpropagation(
            X_batch, Y_batch, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3)

        # Step 4: Update weights
        W1, b1, W2, b2, W3, b3 = update_weights(
            W1, b1, W2, b2, W3, b3,
            dW1, db1, dW2, db2, dW3, db3, learning_rate)

    # Record epoch loss
    _, _, _, _, _, A3_full = forward_pass(X_train, W1, b1, W2, b2, W3, b3)
    epoch_loss = mse_loss(Y_train_oh, A3_full)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch+1}/{epochs} Loss: {epoch_loss:.4f}")


---
## Required Output 1 — Loss Curve

A plot of MSE loss vs. epoch number over the full training run. The curve should visibly decrease, confirming the network is learning.


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, epochs + 1), loss_history,
         marker='o', color='steelblue', linewidth=2, markersize=5)
plt.title("MSE Loss vs Epoch  (MNIST Training)", fontsize=14)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.xticks(range(1, epochs + 1))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()


---
## Required Output 2 — Test Accuracy

Print the final classification accuracy on the 10,000 test images as a percentage.


In [ ]:
_, _, _, _, _, A3_test = forward_pass(X_test, W1, b1, W2, b2, W3, b3)
predicted_labels = np.argmax(A3_test, axis=1)
accuracy = np.mean(predicted_labels == Y_test) * 100.0

print(f"Test Accuracy: {accuracy:.2f}%  ({int(accuracy * 100)} / 10000 correctly classified)")


---
## Required Output 3 — Sample Predictions

Display 10 test images (one per digit class 0–9) with the true label and the network's predicted label shown as the subplot title.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
fig.suptitle("Sample Predictions — One Image per Digit Class (0–9)", fontsize=13, y=1.01)

for digit in range(10):
    ax = axes[digit // 5][digit % 5]

    # Find first test image of this digit class
    idx  = np.where(Y_test == digit)[0][0]
    img  = X_test[idx].reshape(28, 28)
    pred = predicted_labels[idx]

    ax.imshow(img, cmap='gray')
    color = 'green' if pred == digit else 'red'
    ax.set_title(f"True: {digit}  |  Pred: {pred}", color=color,
                 fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=150, bbox_inches='tight')
plt.show()
